# 01 - Quickstart

Minimum viable use of `narratoflow`. Runs **offline** (no API key required) by stacking the free deterministic layers and using `MockProvider` for the LLM-backed extract layer.

What you will learn:
1. How to construct a `Compressor`
2. What a `CompressionResult` actually contains
3. How `Decoder` builds the final downstream prompt
4. How to read the per-layer stats

What this notebook will *not* prove: real cost savings. See `02_token_savings.ipynb` for that — it makes real API calls.


## Setup

In [ ]:
# If installed from PyPI:
#   pip install narratoflow
# If running from a checked-out repo:
#   pip install -e .

from narrato import Compressor, Decoder, list_profiles
from narrato.providers import MockProvider

print("profiles available:")
for p in list_profiles():
    print(f"  {p.name:<16} {p.description}")


## A real-ish source document

Short on purpose — large enough to see the layers act, small enough to read in full.


In [ ]:
SOURCE = '''\
Q3 2026 product strategy meeting, October 14. Present: Alex (PM), Priya (engineering lead),
Marco (design), Sam (data). Alex opened by recapping Q2 metrics: weekly active users grew 18%
to 240,000; retention at day 30 dropped from 32% to 29%.

Priya raised concern that the search rewrite (Project Halibut) is two weeks behind. Marco
noted that the new onboarding screens shipped in beta on October 7 and the cohort that saw
them shows 4 percentage points higher day-7 retention.

Sam presented the funnel analysis: the biggest drop-off is between sign-up and first message,
where 41% of users leave. Alex proposed three Q3 priorities: (1) finish Halibut by November 30,
(2) extend the new onboarding flow to 100% of new users by November 15, (3) ship the message-
draft AI assistant for paid users by December 20.

Priya asked about staffing: she needs one more backend engineer to hit November 30. Alex will
talk to Recruiting tomorrow. Marco committed to delivering the AI-assistant design spec by
October 28. Sam will rerun the funnel after the onboarding rollout completes.
'''

print(f"source length: {len(SOURCE)} chars")


## Compress with just the free layers

No LLM call. Pure CPU. The biggest wins from these two layers are sentence dedupe and frequent-phrase substitution.


In [ ]:
c_free = Compressor(
    provider="openai",                # only used for tokenizer here, not for any call
    source_lang="en",
    layers=["preprocess", "codebook"],
)

free_result = c_free.compress(SOURCE)

print("tokens in :", free_result.stats["input_tokens"])
print("tokens out:", free_result.stats["output_tokens"])
print("ratio    :", round(free_result.stats["ratio"], 3))
print("layers   :", free_result.layers_run)
print("legend   :", free_result.legend)
print()
print("--- compressed text ---")
print(free_result.payload)


## Add the extract layer (MockProvider)

The extract layer is where the **big** wins live, but it costs a small LLM call. To stay offline we wire in `MockProvider` with a hand-written payload that mimics what a real extractor would return.

In a real run you would replace `MockProvider` with `provider="openai"` or `"anthropic"` or `"ollama"`.


In [ ]:
mock = MockProvider(payload={
    "summary": "Q3 2026 product strategy meeting. WAU up 18%, day-30 retention down to 29%.",
    "entities": ["Alex (PM)", "Priya (eng)", "Marco (design)", "Sam (data)", "Project Halibut"],
    "dates": ["October 14", "October 7", "October 28", "November 15", "November 30", "December 20"],
    "claims": [
        "WAU grew 18% to 240,000 in Q2.",
        "Day-30 retention dropped from 32% to 29%.",
        "Halibut is two weeks behind schedule.",
        "New onboarding screens lifted day-7 retention by 4 percentage points.",
        "41% of users drop off between sign-up and first message.",
        "Three Q3 priorities: finish Halibut by Nov 30, extend onboarding to 100% by Nov 15, ship AI assistant for paid users by Dec 20.",
    ],
})

c_full = Compressor(
    provider=mock,
    source_lang="en",
    layers=["preprocess", "codebook", "extract"],
    schema="qa",
)

full_result = c_full.compress(SOURCE)

print("tokens in :", full_result.stats["input_tokens"])
print("tokens out:", full_result.stats["output_tokens"])
print("ratio    :", round(full_result.stats["ratio"], 3))
print("layers   :", full_result.layers_run)
print("extract  :", full_result.stats["extract"])
print()
print("--- compressed payload (JSON) ---")
print(full_result.payload)


## Build the downstream prompt

`Decoder.unpack_prompt` assembles legend + payload + your instruction. This is what you would send to your big model (gpt-4o, claude-opus-4-7, …).


In [ ]:
prompt = Decoder.unpack_prompt(
    full_result,
    instruction="Write a 100-word executive summary of the meeting for the CEO.",
)
print(prompt)


## Side-by-side


In [ ]:
baseline = c_free.stats if False else free_result.stats
extracted = full_result.stats

print(f"{'configuration':<25} {'tokens_out':>12} {'ratio':>8}")
print("-" * 50)
print(f"{'raw source':<25} {baseline['input_tokens']:>12} {1.0:>8.2f}")
print(f"{'free layers only':<25} {baseline['output_tokens']:>12} {baseline['ratio']:>8.2f}")
print(f"{'+ extract (qa schema)':<25} {extracted['output_tokens']:>12} {extracted['ratio']:>8.2f}")


## What's next

- `02_token_savings.ipynb` — same flow, real OpenAI call, measured dollars saved
- `03_long_doc_async.ipynb` — chunked map-reduce + `acompress`, sync vs async wall clock
- `04_custom_schema.ipynb` — define your own Pydantic schema and watch the extractor fill it
